## **Import libraries**

In [ ]:
import os
import random
import asyncio

SEED = 8
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
import pandas as pd

from natsort import natsorted
from collections import defaultdict

import torch
import torch.nn as nn
from transformers import set_seed as transformers_set_seed

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

def seed_everything(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers_set_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

seed_everything(SEED)

from SARVI.config import (
    paths
)
from SARVI.models.schemas import (
    PipelineContext,
    DOCXToJSONSConfig,
    LLMConfig
)
from SARVI.models.losses import (
    MoMLoss
)

from SARVI.data_io.reader import (
    textwrap,
    read_txt_list, read_ann_list,
    read_json_single,
    read_excel_single
)
from SARVI.data_io.writing import (
    write_torch_checkpoint, write_json_extra_docs, write_parquet
)

from SARVI.services.common.llm_loader import (
    load_llm
)
from SARVI.services.common.tree_funcs import (
    load_tree_hierarchical_module
)
from SARVI.services.common.llm_funcs import (
    prompts as prompts_total
)
from SARVI.services.common.ner_funcs import(
    tokenizer_ner, model_ner, initialize_span_ner_model, NER_WINDOW_LABELS
)
from SARVI.services.sync_funcs.ner_funcs import (
    prepare_data as prepare_data_SYNC,
    construct_loaders_ner as construct_loaders_ner_SYNC,
    run_span_nerclassifier as run_span_nerclassifier_SYNC
)

KeyboardInterrupt: 

## **Initialize variables**

In [ ]:
def initialize_variables(ctx: PipelineContext):
    print("0. Initializing variables\n")
    # create_intermediate_folder_name(ctx)

    #################################################################################################################

    if ctx.cie_10_version == "2018":
        df_reference = read_excel_single(ctx, "Diagnosticos_ES2018.xlsx", sheet_name="finales", header=0)
        df_reference = df_reference[['codigo', 'descripcion']].reset_index(drop=True).rename(columns={'codigo': 'Código', 'descripcion': 'Descripción'})
        df_reference["Código"] = (df_reference["Código"].astype(str).str.strip().str.replace("\xa0", "", regex=False))
        df_reference = df_reference[df_reference["Código"].str.match(r"^[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?(?:-[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?)?$")]

    elif ctx.cie_10_version == "2024":
        df_reference = read_excel_single(ctx, "Diagnosticos_ES2024.xlsx", sheet_name="ES2024 Completa + Marcadores", header=0)
        df_reference = df_reference[["Código", "Descripción"]].reset_index(drop=True)
        df_reference["Código"] = (df_reference["Código"].astype(str).str.strip().str.replace("\xa0", "", regex=False))
        df_reference = df_reference[df_reference["Código"].str.match(r"^[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?(?:-[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?)?$")]

    else:
        df_reference = read_excel_single(ctx, "Diagnosticos_ES2026.xlsx", sheet_name="ES2026 Completa + Marcadores", header=0)
        df_reference = df_reference[["Código", "Descripción"]].reset_index(drop=True)
        df_reference["Código"] = (df_reference["Código"].astype(str).str.strip().str.replace("\xa0", "", regex=False))
        df_reference = df_reference[df_reference["Código"].str.match(r"^[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?(?:-[A-Za-z]\d[A-Za-z0-9](?:\.[A-Za-z0-9]+)?)?$")]

    #################################################################################################################

    report_list = natsorted(p for p in (ctx.paths.data_input / ctx.folder_and_archive_name).iterdir() if p.is_file())
    
    if ctx.ussage == "generative":
        llm = load_llm(ctx.llm_config)
        prompt = textwrap.dedent(prompts_total["report_to_data"])
        modelo_ner = None
        node_list = None
        modelo_icd10_head = None
        modelo_icd10_prediction = None
        label2id_ICD10 = None
        id2label_ICD10 = None
        id_no_hs_to_id_hs = None
        icd10_thresholds = None
    else:
        llm = None
        prompt = None
        modelo_ner = [None, None]
        node_list = load_tree_hierarchical_module(df_reference)

        root = node_list["root"]
        root.set_indexes()
        label2id_hs = {key.name: value for key, value in root.node_to_id.items()}
        id2label_hs = {value: key for key, value in label2id_hs.items()}
        label2id_no_hs = {id2label_hs[value.item()]: i for i, value in enumerate(root.leaf_indexes)}
        id2label_no_hs = {value: key for key, value in label2id_no_hs.items()}
        label2id_ICD10 = [label2id_hs, label2id_no_hs]
        id2label_ICD10 = [id2label_hs, id2label_no_hs]
        id_no_hs_to_id_hs = {id_no_hs: label2id_hs[label_name] for label_name, id_no_hs in label2id_no_hs.items()}

        # modelo_icd10_head = [initialize_icd10_hs_head_model(ctx, "ICD10_HS_checkpoint.pt", root), initialize_icd10_no_hs_head_model(ctx, "ICD10_NO_HS_checkpoint.pt", label2id_no_hs, root)]
        # modelo_icd10_prediction = [initialize_icd10_hs_prediction_model(ctx, "ICD10_HS_checkpoint.pt", label2id_hs, root), None]
        modelo_icd10_head = None
        modelo_icd10_prediction = None
        icd10_thresholds = read_json_single(ctx.paths.docs_dir / "thresholds_ICD10.json")
    
    semaforo = asyncio.Semaphore(ctx.MAX_CONCURRENCY)

    config = DOCXToJSONSConfig(
        report_list=report_list,
        prompt=prompt,
        llm=llm,
        semaforo=semaforo,
        modelo_ner=modelo_ner,
        node_list=node_list,
        modelo_icd10_head=modelo_icd10_head,
        modelo_icd10_prediction=modelo_icd10_prediction,
        label2id_ICD10=label2id_ICD10,
        id2label_ICD10=id2label_ICD10,
        id_no_hs_to_id_hs=id_no_hs_to_id_hs,
        icd10_thresholds=icd10_thresholds
    )

    return config

In [ ]:
llm_config = LLMConfig(
    service="vllm",
    model="openai/gpt-oss-20b",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

ctx = PipelineContext(
    ussage="deterministic",
    # folder_and_archive_name="CodiEsp/train",
    folder_and_archive_name="CodiEsp/train",
    llm_config=llm_config,
    paths=paths,
    cie_10_version="2026",
    json_parse=False,
    device = "cuda" if torch.cuda.is_available() else "cpu"
)

config = initialize_variables(ctx)

0. Initializing variables



Hierarchical tree construction: Cleaning data:   0%|          | 0/102426 [00:00<?, ?code/s]

## **Run SPAN NER data creation**

### **Create data**

In [ ]:
print("1.1. Creating NER data // Train data")
dict_data_train = read_txt_list(ctx.paths.data_input / ctx.folder_and_archive_name)
df_data_train = pd.DataFrame(list(dict_data_train.items()), columns=["archivo_origen", "Text"])

dict_ann_train = read_ann_list(ctx.paths.data_input / ctx.folder_and_archive_name)
df_ann_train = pd.DataFrame([{"archivo_origen": file_name, **ann_data}for file_name, ann_data in dict_ann_train.items()])
# df_ann = None

data_prepared_train, all_window_labels_train, file_names_train = prepare_data_SYNC(df_data_train, data_ann=df_ann_train, padding=True, tokenizer=tokenizer_ner, model=model_ner, device=ctx.device, lemma=True)

print("1.1. Creating NER data // Dev data")
dict_data_dev = read_txt_list(ctx.paths.data_input / "CodiEsp/dev")
df_data_dev = pd.DataFrame(list(dict_data_dev.items()), columns=["archivo_origen", "Text"])

dict_ann_dev = read_ann_list(ctx.paths.data_input / "CodiEsp/dev")
df_ann_dev = pd.DataFrame([{"archivo_origen": file_name, **ann_data}for file_name, ann_data in dict_ann_dev.items()])
# df_ann = None

data_prepared_dev, all_window_labels_dev, file_names_dev = prepare_data_SYNC(df_data_dev, data_ann=df_ann_dev, padding=True, tokenizer=tokenizer_ner, model=model_ner, device=ctx.device, lemma=True)

1.1. Creating NER data // Train data


Preparing data for NER prediction: Window slicing and overlapping tokens:   0%|          | 0/500 [00:00<?, ?te…

Token indices sequence length is longer than the specified maximum sequence length for this model (711 > 512). Running this sequence through the model will result in indexing errors


1.1. Creating NER data // Dev data


Preparing data for NER prediction: Window slicing and overlapping tokens:   0%|          | 0/250 [00:00<?, ?te…

> Example of label output

In [5]:
all_window_labels_dev

[[['O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   '1_B-DISO',
   '1_I-DISO',
   '1_I-DISO',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   '{2.1_B-DISO}',
   '{2.1_B-DISO}',
   '{2.1_B-DISO}',
   'O',
   '{2.2_I-DISO}',
   '{2.2_I-DISO}',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   '3_B-DISO',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O

In [6]:
### Example of labels
example_window_labels = all_window_labels_train[2][0]
example_data_prepared = data_prepared_train[2][0]["input_ids"]

for i,(label,token) in enumerate(zip(example_window_labels,tokenizer_ner.convert_ids_to_tokens(example_data_prepared))):
    if label != "O":
        print(f"{label}: {token}")
        if i+1 <= len(example_window_labels) and example_window_labels[i+1] == "O":
            print()

1_B-DISO: ▁e
1_B-DISO: spon
1_B-DISO: dili
1_B-DISO: tis
1_I-DISO: ▁an
1_I-DISO: qui
1_I-DISO: los
1_I-DISO: ante

2_B-DISO;{3.1_B-DISO}: ▁qui
2_B-DISO;{3.1_B-DISO}: ste
2_I-DISO: ▁hep
2_I-DISO: ático

{3.2_I-DISO}: ▁renal

{4.1_B-DISO}: ▁ri
{4.1_B-DISO}: ñó
{4.1_B-DISO}: n

{4.2_I-DISO};5_B-DISO: ▁enfermedad
{4.2_I-DISO}: ▁poli
{4.2_I-DISO}: qu
{4.2_I-DISO}: ístico

6_B-DISO: ▁protein
6_B-DISO: uria

7_B-DISO: ▁protein
7_B-DISO: uria

8_B-DISO: ▁pú
8_B-DISO: r
8_B-DISO: pura

9_B-DISO: ▁her
9_B-DISO: nia
9_I-DISO: ▁um
9_I-DISO: bili
9_I-DISO: cal

10_B-DISO: ▁qui
10_B-DISO: ste
10_I-DISO: ▁hep
10_I-DISO: ático



> Total count of labels

In [7]:
labels_unique = defaultdict(int)
for i in [i for window in all_window_labels_train for sublist in window for i in sublist]:
    labels_unique[i] += 1


bio_order = {"B": 0, "I": 1, "O": 2}
ordered_data = defaultdict(labels_unique.default_factory, sorted(labels_unique.items(), key=lambda x:(x[0].split("-", 1)[1] if "-" in x[0] else x[0], bio_order.get(x[0].split("-", 1)[0], 99))))
ordered_data

defaultdict(int,
            {'8_B-DISO': 1057,
             '8_I-DISO': 698,
             '12_B-DISO': 897,
             '14_B-DISO': 936,
             '1_B-DISO': 788,
             '3_I-DISO': 610,
             '4_B-DISO': 885,
             '4_I-DISO': 670,
             '5_B-DISO': 919,
             '6_I-DISO': 683,
             '9_B-DISO': 1060,
             '9_I-DISO': 601,
             '10_B-DISO': 1021,
             '12_I-DISO': 559,
             '13_B-DISO': 930,
             '1_I-DISO': 701,
             '2_I-DISO': 661,
             '6_B-DISO': 986,
             '7_B-DISO': 1040,
             '10_I-DISO': 753,
             '2_B-DISO': 702,
             '3_B-DISO': 780,
             '5_I-DISO': 735,
             '11_I-DISO': 656,
             '13_I-DISO': 686,
             '15_B-DISO': 1000,
             '16_B-DISO': 833,
             '17_B-DISO': 734,
             '18_B-DISO': 641,
             '18_I-DISO': 306,
             '19_B-DISO': 663,
             '19_I-DISO': 365,
   

### **Create DataLoaders and models**

In [8]:
class SpanCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, ignore_index=-100):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight=weight,
            ignore_index=ignore_index,
            reduction="none"
        )
        self.ignore_index = ignore_index

    def forward(self, logits, targets, weights=None):
        """
        logits:  (N, C)
        targets: (N,)
        weights: optional (N,)
        """
        loss = self.ce(logits, targets)

        valid_mask = targets != self.ignore_index
        loss = loss * valid_mask

        if weights is not None:
            weights = weights.to(device=logits.device, dtype=loss.dtype)
            loss = loss * weights

        return loss.sum() / valid_mask.sum().clamp(min=1)

In [9]:
seed_everything(SEED)

dataset_full_train, data_loader_full_train, label2id, id2label = construct_loaders_ner_SYNC(data_prepared_train, all_window_labels_train, file_names_train, ner_type="span", seed=SEED, ignore_o_labels=True, batch_size=1)
dataset_full_dev, data_loader_full_dev, _, _ = construct_loaders_ner_SYNC(data_prepared_dev, all_window_labels_dev, file_names_dev, ner_type="span", label2id=label2id, id2label=id2label, seed=SEED, ignore_o_labels=True, batch_size=1)

config.modelo_ner[0] = initialize_span_ner_model(ctx, model_ner, len(id2label), id2label, label2id, tokenizer_ner, None, True)
ignore_index = -100

optimizer = torch.optim.AdamW(config.modelo_ner[0].parameters(), lr=1e-5, weight_decay=0.001)
# criterion = MoMLoss(M=512, majority_label=label2id["O"], delta=0.05,
#                     # sentence_loss_fn=DiceLoss(epsilon=1.0, delta=0.01), 
#                     sentence_loss_fn=nn.CrossEntropyLoss(reduction="none", ignore_index=ignore_index),
#                     majority_loss_fn=nn.CrossEntropyLoss(reduction="none", ignore_index=ignore_index),
#                     ignore_index=ignore_index)
criterion = SpanCrossEntropyLoss(ignore_index=ignore_index)

### **Train model**

In [ ]:
seed_everything(SEED)

print("1.2. Running NER model // SPAN")
results = run_span_nerclassifier_SYNC(config.modelo_ner[0], data_loader=data_loader_full_train, device=ctx.device, id2label=id2label, train=True, dev_data_loader=data_loader_full_dev, optimizer=optimizer, criterion=criterion, epochs=25, patience=5, ignore_index=ignore_index, return_predictions=False, ignore_o_labels=True)

write_torch_checkpoint(ctx=ctx, name="NERTraining/Span/CodiEsp/span_nerclassifier_checkpoint", checkpoint=results["best_model_state"])
write_parquet(ctx=ctx, name="NERTraining/Span/CodiEsp/span_nerclassifier_results", data=pd.DataFrame([{k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in results.items() if k not in ["best_model_state", "model_outputs"]}]))
write_json_extra_docs(ctx=ctx, file_name="NERTraining/Span/CodiEsp/id2label_span_nerclassifier", content=id2label)
write_json_extra_docs(ctx=ctx, file_name="NERTraining/Span/CodiEsp/label2id_span_nerclassifier", content=label2id)

1.2. Running NER model // SPAN


Span NER / Training:   0%|          | 0/25 [00:00<?, ?epoch/s]

Span NER / Training epoch 1 / Train set:   0%|          | 0/500 [00:00<?, ?batch/s]

Span NER / Training epoch 1 / Dev set:   0%|          | 0/250 [00:00<?, ?batch/s]


Development classification report:
              precision    recall  f1-score   support

        DISO     0.8827    0.9014    0.8919     13338

   micro avg     0.8827    0.9014    0.8919     13338
   macro avg     0.8827    0.9014    0.8919     13338
weighted avg     0.8827    0.9014    0.8919     13338

Epoch 1/25 | train loss: 0.200261 | dev loss: 0.141052 | Best F1 (macro): 0.8919


...


Development classification report:
              precision    recall  f1-score   support

        DISO     0.9035    0.9201    0.9117     13338

   micro avg     0.9035    0.9201    0.9117     13338
   macro avg     0.9035    0.9201    0.9117     13338
weighted avg     0.9035    0.9201    0.9117     13338

Epoch 12/25 | train loss: 0.012044 | dev loss: 0.326680 | Best F1 (macro): 0.9136


Span NER / Training epoch 13 / Train set:   0%|          | 0/500 [00:00<?, ?batch/s]

Span NER / Training epoch 13 / Dev set:   0%|          | 0/250 [00:00<?, ?batch/s]


Development classification report:
              precision    recall  f1-score   support

        DISO     0.9363    0.8896    0.9123     13338

   micro avg     0.9363    0.8896    0.9123     13338
   macro avg     0.9363    0.8896    0.9123     13338
weighted avg     0.9363    0.8896    0.9123     13338


Early stopping at epoch 13 | Best epoch: 8 | Best dev macro F1: 0.913556


ArrowInvalid: ('Could not convert tensor([[     0,  44543,      8,  ...,   3269,   2409,      2],\n        [     0,   5387,      9,  ...,      4, 146731,      2],\n        [     0,   9723,   9204,  ...,      1,      1,      1]]) with type Tensor: did not recognize Python value type when inferring an Arrow data type', 'Conversion failed for column model_outputs with type object')

In [ ]:
# Use plot styling from seaborn.
sns.set(style='darkgrid')

# Increase the plot size and font size.
sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)

# Plot the learning curve.
plt.plot(results["loss_values"], 'b-o', label="training loss")
plt.plot(results["development_loss_values"], 'r-o', label="validation loss")

# Label the plot.
plt.title("Learning curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

### **Eval model**

In [11]:
print("1.1. Creating NER data // Test data")
# dict_data_test = read_txt_list(ctx.paths.data_input / "CodiEsp/test")
dict_data_test = read_txt_list(ctx.paths.data_input / "CodiEsp/test")
df_data_test = pd.DataFrame(list(dict_data_test.items()), columns=["archivo_origen", "Text"])

# dict_ann_test = read_ann_list(ctx.paths.data_input / "CodiEsp/test")
dict_ann_test = read_ann_list(ctx.paths.data_input / "CodiEsp/test")
df_ann_test = pd.DataFrame([{"archivo_origen": file_name, **ann_data}for file_name, ann_data in dict_ann_test.items()])
# df_ann = None

data_prepared_test, all_window_labels_test, file_names_test = prepare_data_SYNC(df_data_test, data_ann=df_ann_test, padding=True, tokenizer=tokenizer_ner, model=model_ner, device=ctx.device)

1.1. Creating NER data // Test data


Preparing data for NER prediction: Window slicing and overlapping tokens:   0%|          | 0/250 [00:00<?, ?te…

In [15]:
id2label = read_json_single(ctx.paths.docs_dir / "NERTraining/Span/CodiEsp/id2label_span_nerclassifier.json")
label2id = read_json_single(ctx.paths.docs_dir / "NERTraining/Span/CodiEsp/label2id_span_nerclassifier.json")
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: int(k)  for k, v in id2label.items()}

dataset_full_test, data_loader_full_test, _, _ = construct_loaders_ner_SYNC(data_prepared_test, all_window_labels_test, file_names_test, ner_type="span", label2id=label2id, id2label=id2label, seed=SEED, ignore_o_labels=True, batch_size=1)

config.modelo_ner[0] = initialize_span_ner_model(ctx, model_ner, len(id2label), id2label, label2id, tokenizer_ner, "NERTraining/Span/CodiEsp/span_nerclassifier_checkpoint.pt", True)

In [17]:
results = run_span_nerclassifier_SYNC(config.modelo_ner[0], data_loader=data_loader_full_test, device=ctx.device, id2label=id2label, train=False, ignore_o_labels=True)

Span NER / Inference:   0%|          | 0/250 [00:00<?, ?batch/s]


Classification report:
              precision    recall  f1-score   support

        DISO     0.8827    0.9452    0.9129     13209

   micro avg     0.8827    0.9452    0.9129     13209
   macro avg     0.8827    0.9452    0.9129     13209
weighted avg     0.8827    0.9452    0.9129     13209



## **Results to .ann format**

In [18]:
from SARVI.services.common.utils.ann_utils import ner_outputs_to_ann
from SARVI.data_io.writing import write_ann_list

In [19]:
ann_format = ner_outputs_to_ann(results, id2label, data_loader=data_loader_full_test, original_texts=df_data_test, tokenizer=tokenizer_ner)
# write_ann_list(ctx, ann_format, ctx.paths.data_intermediate / ctx.folder_and_archive_name / "span", extra="")
write_ann_list(ctx, ann_format, ctx.paths.data_intermediate / "CodiEsp/test" / "span", extra="")

## **NER results**

In [20]:
from SARVI.services.common.ner_funcs import classification_report_ner

In [21]:
# dict_ann_test_true = read_ann_list(ctx.paths.data_input / "CodiEsp/test")
dict_ann_test_true = read_ann_list(ctx.paths.data_input / "CodiEsp/test")
df_ann_test_true = pd.DataFrame([{"archivo_origen": file_name, **ann_data}for file_name, ann_data in dict_ann_test_true.items()])

# dict_ann_test_pred = read_ann_list(ctx.paths.data_intermediate / ctx.folder_and_archive_name / "span")
dict_ann_test_pred = read_ann_list(ctx.paths.data_intermediate / "CodiEsp/test" / "span")
df_ann_test_pred = pd.DataFrame([{"archivo_origen": file_name, **ann_data}for file_name, ann_data in dict_ann_test_pred.items()])

label2id = read_json_single(ctx.paths.docs_dir / "NERTraining/Span/CodiEsp/label2id_span_nerclassifier.json")

In [22]:
results = classification_report_ner(df_ann_test_true, df_ann_test_pred, [k for k in label2id.keys() if k != "O"])

              precision    recall  f1-score   support   type_1   type_2   type_3   type_4   type_5

        DISO     0.1055    0.6595    0.1819      3665    19248        0        0        0     1248

   micro avg     0.1055    0.6595    0.1819      3665    19248        0        0        0     1248
   macro avg     0.1055    0.6595    0.1819      3665    19248        0        0        0     1248
weighted avg     0.1055    0.6595    0.1819      3665    19248        0        0        0     1248
